In [23]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [24]:
import sys
import os

sys.path.append(os.path.abspath(".."))  

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_validate

import mlflow

import importlib
from src import trainer
importlib.reload(trainer)

from src.preprocessing import build_pipeline
from src.trainer import run_training

mlflow.set_tracking_uri("file:../mlruns")
mlflow.set_experiment("house_prices_models")

<Experiment: artifact_location='file:c:/Users/karen/Documents/proyecto_estadistica_multivariada/notebooks/../mlruns/643315942574303528', creation_time=1776584422785, experiment_id='643315942574303528', last_update_time=1776584422785, lifecycle_stage='active', name='house_prices_models', tags={}, trace_location=None, workspace='default'>

Correr mlflow con python -m mlflow ui

In [25]:
X_train = pd.read_csv("../data/clean/X_train.csv")
y_train = pd.read_csv("../data/clean/y_train.csv").values.ravel()

In [3]:
1460 - len(X_train), len(y_train)

(292, 1168)

## Entrenamiento modelos


In [63]:
from sklearn.linear_model import LinearRegression

result_lr = run_training(
    model_name="linear_regression",
    model=LinearRegression(),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=None,
    use_mlflow=True
)
result_lr

2026/04/19 02:54:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'linear_regression',
 'metrics': {'rmse_mean': np.float64(0.1568804609400624),
  'mse_mean': np.float64(0.025661942637630015),
  'mae_mean': np.float64(0.09222111123691429),
  'r2_mean': np.float64(0.8357476751244615),
  'rmse_real': np.float64(18285.565636485757),
  'mae_real': 11641.215153211811,
  'r2_real': 0.9439418079558423},
 'params': {},
 'model_path': '../artifacts/models\\linear_regression.pkl',
 'metrics_path': '../artifacts/metrics\\linear_regression_metrics.csv'}

In [64]:
from sklearn.linear_model import Ridge

ridge_grid = {
    "model__alpha": [0.1, 1.0, 10.0, 100.0]
}

result_ridge = run_training(
    model_name="ridge",
    model=Ridge(),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=ridge_grid
)

result_ridge

Fitting 5 folds for each of 4 candidates, totalling 20 fits


2026/04/19 02:54:07 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'ridge',
 'metrics': {'rmse_log': np.float64(0.11307623411355457),
  'mse_log': 0.012786234721303401,
  'mae_log': 0.07744931850150925,
  'r2_log': 0.9161241770994373,
  'rmse_real': np.float64(24899.36248344699),
  'mse_real': 619978252.0820874,
  'mae_real': 14118.933265303463,
  'r2_real': 0.8960561630622118},
 'params': {'model__alpha': 10.0},
 'model_path': '../artifacts/models\\ridge.pkl',
 'metrics_path': '../artifacts/metrics\\ridge_metrics.csv'}

In [65]:
from sklearn.linear_model import Lasso


lasso_grid = {
    "model__alpha": [0.0001, 0.001, 0.01, 0.1, 1.0]
}

result_lasso = run_training(
    model_name="lasso",
    model=Lasso(max_iter=10000),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=lasso_grid,
    cv=5
)

result_lasso

Fitting 5 folds for each of 5 candidates, totalling 25 fits


2026/04/19 02:54:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'lasso',
 'metrics': {'rmse_log': np.float64(0.11908523677371102),
  'mse_log': 0.014181293617450816,
  'mae_log': 0.08099132124758361,
  'r2_log': 0.9069727955192008,
  'rmse_real': np.float64(26962.624175443572),
  'mse_real': 726983102.4262142,
  'mae_real': 14806.15449113092,
  'r2_real': 0.878116026164878},
 'params': {'model__alpha': 0.001},
 'model_path': '../artifacts/models\\lasso.pkl',
 'metrics_path': '../artifacts/metrics\\lasso_metrics.csv'}

In [12]:
from sklearn.tree import DecisionTreeRegressor

cart_grid = {
    "model__max_depth": [None,2, 3, 5],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4, 10, 20],
    "model__max_features": [None, "sqrt", "log2"]

}

result_cart = run_training(
    model_name="decision_tree",
    model=DecisionTreeRegressor(random_state=42),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=cart_grid,
    cv=5
)

result_cart

Fitting 5 folds for each of 180 candidates, totalling 900 fits


2026/04/24 12:07:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'decision_tree',
 'metrics': {'rmse_log': np.float64(0.1461506777328072),
  'mse_log': 0.021360020601758865,
  'mae_log': 0.1031480921485695,
  'r2_log': 0.8598814002561289,
  'rmse_real': np.float64(30825.796748602774),
  'mse_real': 950229745.1861694,
  'mae_real': 18784.173922895072,
  'r2_real': 0.8406871122408506},
 'params': {'model__max_depth': None,
  'model__max_features': None,
  'model__min_samples_leaf': 20,
  'model__min_samples_split': 2},
 'model_path': '../artifacts/models\\decision_tree.pkl',
 'metrics_path': '../artifacts/metrics\\decision_tree_metrics.csv'}

In [41]:
from sklearn.ensemble import RandomForestRegressor

rf_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [5, 10, 15],
    "model__min_samples_leaf": [5, 10, 20, 30],
    "model__min_samples_split": [10, 20, 50],
    "model__max_features": ["sqrt", "log2", 0.5]
}

result_rf = run_training(
    model_name="random_forest",
    model=RandomForestRegressor(random_state=42),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=rf_grid
)
result_rf

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


2026/04/24 15:21:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'random_forest',
 'metrics': {'rmse_log': np.float64(0.09433957124729837),
  'mse_log': 0.008899954703124085,
  'mae_log': 0.0606545002984272,
  'r2_log': 0.9416176035577916,
  'rmse_real': np.float64(20643.84350952937),
  'mse_real': 426168274.84593797,
  'mae_real': 11056.820272547535,
  'r2_real': 0.9285498071587526},
 'params': {'model__max_depth': 10,
  'model__max_features': 0.5,
  'model__min_samples_leaf': 5,
  'model__min_samples_split': 10,
  'model__n_estimators': 200},
 'model_path': '../artifacts/models\\random_forest.pkl',
 'metrics_path': '../artifacts/metrics\\random_forest_metrics.csv'}

In [52]:
from xgboost import XGBRegressor

xgb_grid = {
    "model__n_estimators": [200],
    "model__max_depth": [3, 4],
    "model__learning_rate": [0.03, 0.05],
    "model__subsample": [0.6, 0.8],
    "model__colsample_bytree": [0.5, 0.8],
    "model__min_child_weight": [5, 10],
    "model__reg_alpha": [0, 0.1],
    "model__reg_lambda": [1, 5]
}

result_xgb = run_training(
    model_name="xgboost",
    model=XGBRegressor(
        random_state=42,
        verbosity=0
    ),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=xgb_grid
)

result_xgb

Fitting 5 folds for each of 128 candidates, totalling 640 fits


C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
2026/04/24 15:26:32 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'xgboost',
 'metrics': {'rmse_log': np.float64(0.08278402549035269),
  'mse_log': 0.006853194876387363,
  'mae_log': 0.05584497952332727,
  'r2_log': 0.9550440475805442,
  'rmse_real': np.float64(17349.199312705747),
  'mse_real': 300994716.7919895,
  'mae_real': 10025.354759738875,
  'r2_real': 0.949536059279028},
 'params': {'model__colsample_bytree': 0.5,
  'model__learning_rate': 0.05,
  'model__max_depth': 4,
  'model__min_child_weight': 10,
  'model__n_estimators': 200,
  'model__reg_alpha': 0,
  'model__reg_lambda': 5,
  'model__subsample': 0.6},
 'model_path': '../artifacts/models\\xgboost.pkl',
 'metrics_path': '../artifacts/metrics\\xgboost_metrics.csv'}

In [78]:
from lightgbm import LGBMRegressor

lgbm_grid = {
    "model__n_estimators": [200],
    "model__learning_rate": [0.03, 0.05],
    "model__max_depth": [5, 8],
    "model__num_leaves": [15, 31],
    "model__min_data_in_leaf": [20, 50],
    "model__feature_fraction": [0.6, 0.8],
    "model__bagging_fraction": [0.6, 0.8],
    "model__bagging_freq": [1],
    "model__lambda_l1": [0, 0.1],
    "model__lambda_l2": [1, 5]
}

result_lgbm = run_training(
    model_name="lightgbm",
    model=LGBMRegressor(random_state=42),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=lgbm_grid
)

result_lgbm

Fitting 5 folds for each of 256 candidates, totalling 1280 fits
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning]

C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1


2026/04/24 16:05:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'lightgbm',
 'metrics': {'rmse_log': np.float64(0.07562293490152651),
  'mse_log': 0.005718828283120517,
  'mae_log': 0.05067168869186062,
  'r2_log': 0.9624853259204952,
  'rmse_real': np.float64(15764.635021028926),
  'mse_real': 248523717.34625167,
  'mae_real': 9094.54561565571,
  'r2_real': 0.9583332017465809},
 'params': {'model__bagging_fraction': 0.8,
  'model__bagging_freq': 1,
  'model__feature_fraction': 0.6,
  'model__lambda_l1': 0,
  'model__lambda_l2': 1,
  'model__learning_rate': 0.05,
  'model__max_depth': 5,
  'model__min_data_in_leaf': 20,
  'model__n_estimators': 200,
  'model__num_leaves': 15},
 'model_path': '../artifacts/models\\lightgbm.pkl',
 'metrics_path': '../artifacts/metrics\\lightgbm_metrics.csv'}

In [55]:
from catboost import CatBoostRegressor

catboost_grid = {
    "model__iterations": [500],
    "model__depth": [3, 4, 5],
    "model__learning_rate": [0.03],
    
    "model__l2_leaf_reg": [10, 20],
    "model__min_data_in_leaf": [20, 50],
    
    "model__subsample": [0.6, 0.8],
    "model__colsample_bylevel": [0.6, 0.8],
    
    "model__random_strength": [5, 10]
}

result_catboost = run_training(
    model_name="catboost",
    model=CatBoostRegressor(
        verbose=0,
        random_state=42,
        iterations=2000,
        learning_rate=0.03,
        depth=4,
        early_stopping_rounds=50
    ),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=catboost_grid
)

result_catboost

Fitting 5 folds for each of 96 candidates, totalling 480 fits


2026/04/24 15:39:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'catboost',
 'metrics': {'rmse_log': np.float64(0.09844225278229089),
  'mse_log': 0.009690877132852459,
  'mae_log': 0.07127112142371929,
  'r2_log': 0.9364292685170269,
  'rmse_real': np.float64(19882.948534924333),
  'mse_real': 395331642.4424497,
  'mae_real': 12807.700872503896,
  'r2_real': 0.9337197915565831},
 'params': {'model__colsample_bylevel': 0.8,
  'model__depth': 5,
  'model__iterations': 500,
  'model__l2_leaf_reg': 10,
  'model__learning_rate': 0.03,
  'model__min_data_in_leaf': 20,
  'model__random_strength': 5,
  'model__subsample': 0.6},
 'model_path': '../artifacts/models\\catboost.pkl',
 'metrics_path': '../artifacts/metrics\\catboost_metrics.csv'}

In [56]:
from sklearn.neural_network import MLPRegressor

mlp_grid = {
    "model__hidden_layer_sizes": [(32,), (64,), (64, 32)],
    "model__alpha": [0.001, 0.01],
    "model__learning_rate_init": [0.0005, 0.001]
}

result_mlp = run_training(
    model_name="mlp",
    model=MLPRegressor(
        max_iter=500,
        random_state=42,
         early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20
    ),
    pipeline_builder=build_pipeline,
    X=X_train,
    y=y_train,
    param_grid=mlp_grid
)

result_mlp

Fitting 5 folds for each of 12 candidates, totalling 60 fits


C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
2026/04/24 15:40:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


{'model_name': 'mlp',
 'metrics': {'rmse_log': np.float64(0.14969487125957384),
  'mse_log': 0.022408554481420387,
  'mae_log': 0.11131837205989487,
  'r2_log': 0.8530031719181804,
  'rmse_real': np.float64(26669.229768224468),
  'mse_real': 711247816.4303501,
  'mae_real': 19027.367851012114,
  'r2_real': 0.880754160641742},
 'params': {'model__alpha': 0.01,
  'model__hidden_layer_sizes': (32,),
  'model__learning_rate_init': 0.001},
 'model_path': '../artifacts/models\\mlp.pkl',
 'metrics_path': '../artifacts/metrics\\mlp_metrics.csv'}

## Comparar los modelos


In [89]:
X_test = pd.read_csv("../data/clean/X_test.csv")
y_test = pd.read_csv("../data/clean/y_test.csv").values.ravel()

In [90]:
def evaluate_saved_models(
    model_paths_dict,
    X_test,
    y_test,
    inverse_log=True
):
    
    import joblib
    import numpy as np
    import pandas as pd
    import os
    from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
    
    y_test = np.array(y_test).ravel()
    
    results = []
    
    for name, path in model_paths_dict.items():
        
        if not os.path.exists(path):
            print(f"Modelo no encontrado: {path}")
            continue
        
        try:
            model = joblib.load(path)
            y_pred = model.predict(X_test)
            
            if inverse_log:
                y_pred = np.expm1(y_pred)
                y_true = np.expm1(y_test)
            else:
                y_true = y_test
            
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mse = mean_squared_error(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            r2 = r2_score(y_true, y_pred)
            
            results.append({
                "model": name,
                "rmse_test": rmse,
                "mse_test": mse,
                "mae_test": mae,
                "r2_test": r2
            })
        
        except Exception as e:
            print(f"Error en modelo {name}: {e}")
            continue
    
    results_df = pd.DataFrame(results)
    
    if not results_df.empty:
        results_df = results_df.sort_values("rmse_test").reset_index(drop=True)
    
    return results_df

In [91]:

models_dir = "../artifacts/models"

model_paths = {
    f.replace(".pkl", ""): os.path.join(models_dir, f)
    for f in os.listdir(models_dir)
    if f.endswith(".pkl")
}

model_paths



{'catboost': '../artifacts/models\\catboost.pkl',
 'decision_tree': '../artifacts/models\\decision_tree.pkl',
 'lasso': '../artifacts/models\\lasso.pkl',
 'lightgbm': '../artifacts/models\\lightgbm.pkl',
 'linear_regression': '../artifacts/models\\linear_regression.pkl',
 'mlp': '../artifacts/models\\mlp.pkl',
 'random_forest': '../artifacts/models\\random_forest.pkl',
 'ridge': '../artifacts/models\\ridge.pkl',
 'xgboost': '../artifacts/models\\xgboost.pkl'}

In [92]:
results_test = evaluate_saved_models(
    model_paths,
    X_test,
    y_test
)

results_test

C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\karen\Ap

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1


C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,model,rmse_test,mse_test,mae_test,r2_test
0,linear_regression,22787.394156,5.192653e+08,15103.852740,0.932302
1,ridge,25160.128667,6.330321e+08,16193.925688,0.917470
2,lasso,25890.657447,6.703261e+08,16210.296841,0.912608
3,lightgbm,28455.403310,8.097100e+08,15824.584424,0.894436
4,xgboost,28964.017415,8.389143e+08,15911.971894,0.890629
5,catboost,29981.389045,8.988837e+08,16545.696888,0.882810
6,random_forest,32521.545421,1.057651e+09,17522.539370,0.862111
7,decision_tree,39096.491060,1.528536e+09,23478.232535,0.800721
8,mlp,40511.805077,1.641206e+09,25842.590087,0.786032


In [93]:
results_test.to_csv("../data/results/test_results.csv", index=False)
print("Resultados de test guardados en ../data/results/test_results.csv")

Resultados de test guardados en ../data/results/test_results.csv


In [94]:
results_test_log = evaluate_saved_models(
    model_paths,
    X_test,
    y_test,
    inverse_log=False
)

results_test_log

C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\karen\Ap

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1


C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,model,rmse_test,mse_test,mae_test,r2_test
0,linear_regression,0.126692,0.016051,0.088325,0.913988
1,ridge,0.133531,0.017830,0.093008,0.904451
2,xgboost,0.134292,0.018034,0.087439,0.903358
3,lightgbm,0.136642,0.018671,0.088456,0.899947
4,catboost,0.137363,0.018868,0.090745,0.898889
5,lasso,0.138649,0.019223,0.093163,0.896986
6,random_forest,0.150934,0.022781,0.097393,0.877923
7,decision_tree,0.182627,0.033353,0.132242,0.821272
8,mlp,0.226108,0.051125,0.154150,0.726036


In [95]:
def select_best_model(results_df, metric="rmse_test"):
    """
    Selecciona el mejor modelo según métrica
    
    Parameters
    ----------
    results_df : pd.DataFrame
    metric : str
        Métrica a minimizar (rmse_test recomendado)
    
    Returns
    -------
    best_row : pd.Series
    best_model_name : str
    """
    
    if metric not in results_df.columns:
        raise ValueError(f"{metric} no está en el DataFrame")
    
    # ordenar por métrica (menor es mejor)
    best_row = results_df.sort_values(metric).iloc[0]
    
    best_model_name = best_row["model"]
    
    print("🏆 Mejor modelo:")
    print(f"Modelo: {best_model_name}")
    print(f"{metric}: {best_row[metric]:.4f}")
    
    return best_row, best_model_name

In [96]:
# evaluar todos los modelos
results_test = evaluate_saved_models(
    model_paths,
    X_test,
    y_test
)

# seleccionar mejor
best_row, best_model_name = select_best_model(results_test)

C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
C:\Users\karen\Ap

[LightGBM] [Warning] min_data_in_leaf is set=20, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=20
[LightGBM] [Warning] feature_fraction is set=0.6, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.6
[LightGBM] [Warning] lambda_l1 is set=0, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0
[LightGBM] [Warning] lambda_l2 is set=1, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1


C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


🏆 Mejor modelo:
Modelo: linear_regression
rmse_test: 22787.3942


C:\Users\karen\AppData\Roaming\Python\Python314\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [97]:
import joblib

best_model_path = model_paths[best_model_name]
best_model = joblib.load(best_model_path)

In [98]:
joblib.dump(best_model, "../artifacts/final_model.pkl")

['../artifacts/final_model.pkl']